**Note:** Not consumed by Notebook 11; scaling is performed in-memory in NB11 per contract (GSN sequence-wise, TBSN channel-wise train-only).


# Notebook 08 — Final Normalization and Scaling (NaN-aware, current-run audited)

## Purpose
Apply the four report scaling pipelines to the sequence-normalized outputs of Notebook 06 (**GSN**) and Notebook 07 (**TBSN**), and hand clean, auditable artifacts to Notebook 09.

This notebook is **only** the final scaling stage. It does not modify GSN, TBSN, missing-data handling, trial definitions, subject IDs, labels, the 751-frame length or the 180-feature contract. It creates **no** train/validation/test split, performs **no** imputation (no zero-fill, no median-fill) and trains **no** model.

## Data contract (inherited, verified at run time)
| Item | Value |
|---|---|
| Sensor features | 180 = 135 linear (`lin__*`) + 45 angular (`ang__*`) (60 nodes × 3) |
| Frames per trial | 751 |
| Sensor matrix `X` | 751 × 180 — never contains `frame`, `time`, subject, condition, medication or labels |
| Metadata columns | `frame`, `time` kept as separate leading CSV columns |
| Missing values | `NaN` in the source CSV (upstream intentionally keeps NaN) |

## Missing-value / mask convention
Notebook 08 reads only the GSN/TBSN trial CSVs, their manifests and the feature provenance table; no separate upstream mask file is loaded here. The validity information therefore **is the finiteness pattern of the source CSV of the same branch**:

`M[t,f] = 1` if `np.isfinite(X_source[t,f])`, else `0`  (shape 751 × 180, saved as `uint8` `.npy` per trial and branch).

* Statistics use **finite values only**.
* Scaling is applied **only to finite values**; missing positions stay `NaN` (never `0`).
* `±Inf` is not a legitimate missing-value code; it is treated as invalid, written as `NaN`, `M = 0`, and counted in the audit.
* The mask is never shifted: validation re-derives it from the source and compares.

## The four branches (identity = Experiment A–D labels of the report tables)
| Experiment | Branch name | Source | Scaling | Granularity |
|---|---|---|---|---|
| Exp_A | `exp_A_gsn_minmax`  | GSN  | Min-Max | **sequence-wise (per walking trial)** |
| Exp_B | `exp_B_gsn_zscore`  | GSN  | Z-score | **sequence-wise (per walking trial)** |
| Exp_C | `exp_C_tbsn_minmax` | TBSN | Min-Max | **channel-wise** |
| Exp_D | `exp_D_tbsn_zscore` | TBSN | Z-score | **channel-wise** |

> **Final, authoritative design: GSN → sequence-wise, TBSN → channel-wise.** This is fixed in the code (no runtime option can reverse it). An earlier version had the granularities reversed; that has been corrected, and `REPORT_ALIGNMENT_DF` is a mandatory final gate that fails if any of the four definitions deviates.

### Definitions
* **Sequence-wise** (per trial, per feature, finite values of that trial only)
  Min-Max: `(x − xmin)/(xmax − xmin)`  Z-score: `(x − mean)/std`, population std (`ddof = 0`).
* **Channel-wise** (per feature, finite values pooled over **all frames of all eligible trials of that source**, report-reproduction scope — *not* train-only; no split exists at this stage). The same formulas are applied with those pooled statistics.
* **Constant channel / constant trial-feature** (`xmax == xmin` over valid values, including a single valid value): all valid values map to **0**, `NaN`s preserved, no division by zero, no `Inf`. Same rule in every branch. A feature with **no** valid value stays all-`NaN` and is flagged `no_valid_values_flag`.

> **Leakage scope.** Channel-wise statistics here use all eligible trials of the source (report reproduction). They are **not** leakage-safe for predictive evaluation. Train-only fitting, if required, belongs to a later experimental notebook or a separate branch.

## Run isolation
Every execution creates a **fresh** run id and fresh directories (`scaled_features_run_<RUN_ID>`, `scalers_run_<RUN_ID>`). Previous runs are never deleted or reused. Every validation reads **only** `CURRENT_PROCESSED_DIR` / `CURRENT_SCALERS_DIR`, enforced by a path guard.

## Trial and run statuses
* Trial `COMPLETED` — scaled, written, validated.
* Trial `EXCLUDED` — legitimately cannot be scaled (this notebook's only criterion: **no finite sensor value at all**, `MIN_FINITE_VALUES_PER_TRIAL = 1`). Logged in `_audit/excluded_trials.csv`. Having *some* missing values is **not** a reason to exclude.
* Trial `FAILED` — software/contract failure (missing file, wrong shape, wrong columns, exception).
* Run `READY` — all gates pass, 0 failures, 0 exclusions.
* Run `COMPLETED_WITH_EXCLUSIONS` — all gates pass, 0 failures, ≥1 documented exclusion (still a successful run).
* Run `NOT_READY` — any gate fails or any trial failed. Exclusions alone never cause `NOT_READY`.

In [1]:
# 0. Configuration, fresh run identity and CURRENT-run directories (single source of truth).
import hashlib
import json
import re
import sys
import time
import traceback
import uuid
import warnings
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd

NOTEBOOK_ID = "08_Final_Normalization_and_Scaling"
NOTEBOOK_VERSION = "1.0-READY"

# ---- Data contract inherited from Notebooks 01-07 (verified, never changed here) ----
EXPECTED_FRAMES = 751
EXPECTED_FEATURES = 180
EXPECTED_LINEAR = 135
EXPECTED_ANGULAR = 45
META_COLUMNS = ["frame", "time"]
FORBIDDEN_FEATURE_NAMES = {
    "frame", "time", "subject_id", "subject", "condition", "medication",
    "label", "labels", "group", "trial_key", "walk", "trial",
}

# A trial is legitimately excluded only if it has fewer finite sensor values than this.
MIN_FINITE_VALUES_PER_TRIAL = 1

# ---- Numerical tolerances (documented, deterministic) ----
FORMULA_RTOL, FORMULA_ATOL = 1e-9, 1e-12   # output vs. formula(source, stored scaler)
STATS_RTOL, STATS_ATOL = 1e-9, 1e-12       # stored scaler vs. independent recomputation
RANGE_TOL = 1e-9                           # Min-Max range checks
MOMENT_TOL = 1e-7                          # Z-score mean / std checks

# ---- Descriptive label recorded in manifests (the design itself is hard-coded in the branch table) ----
DESIGN_LABEL = "final_design__GSN_sequence_wise__TBSN_channel_wise"   # descriptive label only, NOT a switch


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "configs" / "data_config.yaml").is_file() and (p / "src").is_dir():
            return p
        nested = p / "Parkinson_Gait_CIGTNet"
        if (nested / "configs" / "data_config.yaml").is_file() and (nested / "src").is_dir():
            return nested
    return cwd.parent if cwd.name == "notebooks" else cwd


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
NORMALIZED_DIR = DATA_DIR / "normalized"
GSN_DIR = NORMALIZED_DIR / "gsn"
TBSN_DIR = NORMALIZED_DIR / "tbsn"
METADATA_DIR = DATA_DIR / "metadata"

# Legacy / canonical locations: defined ONLY so that the notebook can prove it never reads them.
LEGACY_PROCESSED_DIR = DATA_DIR / "processed" / "scaled_features"
LEGACY_SCALERS_DIR = METADATA_DIR / "scalers"

# ---- Fresh run identity (created at the very start; previous runs are never touched) ----
RUN_STARTED_UTC = datetime.now(timezone.utc)
RUN_TIMESTAMP = RUN_STARTED_UTC.isoformat(timespec="seconds")
RUN_ID = RUN_STARTED_UTC.strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6]

CURRENT_PROCESSED_DIR = DATA_DIR / "processed" / f"scaled_features_run_{RUN_ID}"
CURRENT_SCALERS_DIR = METADATA_DIR / f"scalers_run_{RUN_ID}"
CURRENT_MASKS_DIR = CURRENT_PROCESSED_DIR / "_masks"
CURRENT_AUDIT_DIR = CURRENT_PROCESSED_DIR / "_audit"
RUN_MANIFEST_PATH = CURRENT_PROCESSED_DIR / "run_manifest.json"
LATEST_RUN_POINTER = DATA_DIR / "processed" / "LATEST_SCALING_RUN.json"

for d in [CURRENT_PROCESSED_DIR, CURRENT_SCALERS_DIR, CURRENT_MASKS_DIR, CURRENT_AUDIT_DIR]:
    d.mkdir(parents=True, exist_ok=False)      # exist_ok=False: a run id can never be reused

# Aliases required by the audit contract: everything downstream uses the CURRENT run.
RUN_PROCESSED_DIR = CURRENT_PROCESSED_DIR
RUN_SCALERS_DIR = CURRENT_SCALERS_DIR

ACCESSED_CURRENT_PATHS = set()


def current_path(base: Path, *parts) -> Path:
    """Path guard: every read/write of outputs or scalers must stay inside the CURRENT run dirs."""
    p = Path(base, *parts).resolve()
    allowed = [CURRENT_PROCESSED_DIR.resolve(), CURRENT_SCALERS_DIR.resolve()]
    if not any(p == a or a in p.parents for a in allowed):
        raise RuntimeError(f"Path escapes the current run directories: {p}")
    for legacy in [LEGACY_PROCESSED_DIR.resolve(), LEGACY_SCALERS_DIR.resolve()]:
        if p == legacy or legacy in p.parents:
            raise RuntimeError(f"Legacy/canonical artifact access is forbidden: {p}")
    ACCESSED_CURRENT_PATHS.add(str(p))
    return p


def rel(p) -> str:
    p = Path(p).resolve()
    try:
        return p.relative_to(PROJECT_ROOT.resolve()).as_posix()
    except ValueError:
        return p.as_posix()


STAGE_ERRORS: Dict[str, str] = {}
STAGE_SECONDS: Dict[str, float] = {}


def run_stage(name: str, fn) -> bool:
    """Run a stage; an unexpected exception is recorded (and later forces NOT_READY), never hidden."""
    t0 = time.time()
    try:
        fn()
        ok = True
    except Exception:
        STAGE_ERRORS[name] = traceback.format_exc()
        print(f"[STAGE FAILED] {name}\n{STAGE_ERRORS[name]}")
        ok = False
    STAGE_SECONDS[name] = round(time.time() - t0, 2)
    return ok


print(f"NOTEBOOK        : {NOTEBOOK_ID} v{NOTEBOOK_VERSION}")
print(f"PROJECT_ROOT    : {PROJECT_ROOT}")
print(f"RUN_ID          : {RUN_ID}   ({RUN_TIMESTAMP})")
print(f"CURRENT_PROCESSED_DIR : {CURRENT_PROCESSED_DIR}")
print(f"CURRENT_SCALERS_DIR   : {CURRENT_SCALERS_DIR}")
print(f"Granularity profile   : {DESIGN_LABEL}")
print("No train/validation/test split, no imputation, no model training in this notebook.")

NOTEBOOK        : 08_Final_Normalization_and_Scaling v1.0-READY
PROJECT_ROOT    : C:\Users\lab\Minor_project\Parkinson_gait
RUN_ID          : 20261009_142420_27f01a   (2026-10-09T14:24:20+00:00)
CURRENT_PROCESSED_DIR : C:\Users\lab\Minor_project\Parkinson_gait\data\processed\scaled_features_run_20261009_142420_27f01a
CURRENT_SCALERS_DIR   : C:\Users\lab\Minor_project\Parkinson_gait\data\metadata\scalers_run_20261009_142420_27f01a
Granularity profile   : final_design__GSN_sequence_wise__TBSN_channel_wise
No train/validation/test split, no imputation, no model training in this notebook.


In [2]:
# 1. Load upstream manifests and runtime feature provenance (inputs are only READ).
SOURCES = ["GSN", "TBSN"]
SOURCE_DIRS = {"GSN": GSN_DIR, "TBSN": TBSN_DIR}
GSN_MANIFEST = METADATA_DIR / "gsn_manifest.csv"
TBSN_MANIFEST = METADATA_DIR / "tbsn_manifest.csv"
PROVENANCE = METADATA_DIR / "merged_feature_provenance.csv"

INPUT_ERRORS: List[str] = []
gsn_manifest = tbsn_manifest = prov = None
feature_names: List[str] = []
common_keys: List[str] = []
trial_meta = pd.DataFrame()
FEATURE_HASH = ""
N_LINEAR = N_ANGULAR = 0


def _input_error(msg: str) -> None:
    INPUT_ERRORS.append(msg)


for required in [GSN_MANIFEST, TBSN_MANIFEST, PROVENANCE]:
    if not required.is_file():
        _input_error(f"Missing required upstream artifact: {required}")
for src_name, d in SOURCE_DIRS.items():
    if not d.is_dir():
        _input_error(f"Missing {src_name} trial directory: {d}")

if not INPUT_ERRORS:
    gsn_manifest = pd.read_csv(GSN_MANIFEST)
    tbsn_manifest = pd.read_csv(TBSN_MANIFEST)
    prov = pd.read_csv(PROVENANCE)
    for name, m in [("GSN", gsn_manifest), ("TBSN", tbsn_manifest)]:
        missing_cols = {"trial_key", "subject_id"} - set(m.columns)
        if missing_cols:
            _input_error(f"{name} manifest lacks columns {sorted(missing_cols)}")
        elif m["trial_key"].astype(str).duplicated().any():
            _input_error(f"{name} manifest contains duplicate trial_key values")
    if "merged_feature_name" not in prov.columns:
        _input_error("merged_feature_provenance.csv lacks 'merged_feature_name'")

if not INPUT_ERRORS:
    names = prov["merged_feature_name"].astype(str)
    feature_names = names[names.str.startswith(("lin__", "ang__"))].tolist()
    N_LINEAR = sum(n.startswith("lin__") for n in feature_names)
    N_ANGULAR = sum(n.startswith("ang__") for n in feature_names)
    FEATURE_HASH = hashlib.sha256("\n".join(feature_names).encode("utf-8")).hexdigest()
    if len(feature_names) != len(set(feature_names)):
        _input_error("Duplicate signal feature names in provenance")
    if (len(feature_names), N_LINEAR, N_ANGULAR) != (EXPECTED_FEATURES, EXPECTED_LINEAR, EXPECTED_ANGULAR):
        _input_error(f"Feature contract violated: total={len(feature_names)} lin={N_LINEAR} ang={N_ANGULAR} "
                     f"(expected {EXPECTED_FEATURES}/{EXPECTED_LINEAR}/{EXPECTED_ANGULAR})")
    if FORBIDDEN_FEATURE_NAMES & {n.lower() for n in feature_names}:
        _input_error("Metadata/label-like names found inside the sensor feature list")

    gsn_keys = set(gsn_manifest["trial_key"].astype(str))
    tbsn_keys = set(tbsn_manifest["trial_key"].astype(str))
    gsn_only, tbsn_only = sorted(gsn_keys - tbsn_keys), sorted(tbsn_keys - gsn_keys)
    common_keys = sorted(gsn_keys & tbsn_keys)
    if not common_keys:
        _input_error("No common GSN/TBSN trial keys")
    if gsn_only or tbsn_only:
        _input_error(f"GSN/TBSN inventories differ (GSN-only={len(gsn_only)}, TBSN-only={len(tbsn_only)})")

    g = gsn_manifest.assign(trial_key=gsn_manifest["trial_key"].astype(str)).set_index("trial_key")
    t = tbsn_manifest.assign(trial_key=tbsn_manifest["trial_key"].astype(str)).set_index("trial_key")
    shared = [k for k in common_keys if k in g.index and k in t.index]
    subj_mismatch = [k for k in shared if str(g.at[k, "subject_id"]) != str(t.at[k, "subject_id"])]
    if subj_mismatch:
        _input_error(f"subject_id differs between GSN and TBSN manifests for {len(subj_mismatch)} trials "
                     f"(e.g. {subj_mismatch[0]})")

if not INPUT_ERRORS:
    _KEY_RE = re.compile(r"^(?P<subject>.+?)_(?P<condition>[A-Za-z]+)_walk_(?P<walk>\d+)$")

    def _first(row, names):
        for n in names:
            if n in row.index and pd.notna(row[n]):
                return str(row[n])
        return None

    rows = []
    for k in common_keys:
        row = g.loc[k]
        m = _KEY_RE.match(k)
        walk = _first(row, ["walk", "walk_id", "walk_number", "walk_index"]) or (m.group("walk") if m else "NA")
        rows.append({
            "trial_key": k,
            "subject_id": str(row["subject_id"]),
            "condition": _first(row, ["condition", "medication_state", "med_state", "medication"])
                         or (m.group("condition") if m else "NA"),
            "walk": walk,
            "trial": _first(row, ["trial", "trial_id", "trial_number"]) or walk,   # falls back to walk index
        })
    trial_meta = pd.DataFrame(rows)

INPUT_OK = not INPUT_ERRORS
TRIAL_META_BY_KEY = trial_meta.set_index("trial_key").to_dict("index") if INPUT_OK else {}

print("=== UPSTREAM RECONCILIATION ===")
if INPUT_OK:
    print(f"GSN trials: {len(gsn_keys)} | TBSN trials: {len(tbsn_keys)} | common: {len(common_keys)}")
    print(f"Unique subjects: {trial_meta['subject_id'].nunique()}")
    print(f"Signal features: {len(feature_names)}  (linear {N_LINEAR}, angular {N_ANGULAR})")
    print(f"Feature hash (sha256): {FEATURE_HASH[:16]}...")
else:
    print("INPUT CONTRACT VIOLATIONS (processing will be skipped; final status will be NOT_READY):")
    for e in INPUT_ERRORS:
        print("  -", e)

=== UPSTREAM RECONCILIATION ===
GSN trials: 881 | TBSN trials: 881 | common: 881
Unique subjects: 26
Signal features: 180  (linear 135, angular 45)
Feature hash (sha256): dc8ff94b33fd2d7e...


In [3]:
# 1b. Create Authoritative Trial-Level Split Manifest
import numpy as np
import pandas as pd

TRIAL_SPLIT_MANIFEST = METADATA_DIR / "trial_split_manifest.csv"

if INPUT_OK:
    rng = np.random.default_rng(42)
    splits = []
    for (subj, cond), group in trial_meta.groupby(['subject_id', 'condition']):
        n = len(group)
        tks = group['trial_key'].values.copy()
        rng.shuffle(tks)
        
        n_val = int(round(n * 0.15))
        n_test = int(round(n * 0.15))
        
        if n > 0 and n_val + n_test >= n:
            n_val = max(0, n_val - 1)
            n_test = max(0, n_test - 1)
            if n_val + n_test >= n:
                n_val = 0; n_test = 0
                
        val_keys = tks[:n_val]
        test_keys = tks[n_val:n_val+n_test]
        train_keys = tks[n_val+n_test:]
        
        for k in train_keys: splits.append((k, 'TRAIN'))
        for k in val_keys: splits.append((k, 'VALIDATION'))
        for k in test_keys: splits.append((k, 'TEST'))
        
    df_split = pd.DataFrame(splits, columns=['trial_key', 'split'])
    trial_meta = trial_meta.merge(df_split, on='trial_key')
    
    trial_meta.to_csv(TRIAL_SPLIT_MANIFEST, index=False)
    print(f"Saved trial-level split manifest to {TRIAL_SPLIT_MANIFEST}")
    print(trial_meta['split'].value_counts())
    
    SPLIT_MAP = dict(zip(trial_meta['trial_key'], trial_meta['split']))
else:
    SPLIT_MAP = {}



Saved trial-level split manifest to C:\Users\lab\Minor_project\Parkinson_gait\data\metadata\trial_split_manifest.csv
split
TRAIN         619
TEST          131
VALIDATION    131
Name: count, dtype: int64


C:\Users\lab\AppData\Local\Temp\ipykernel_8004\4237336070.py:13: UserWarning: you are shuffling a 'StringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  rng.shuffle(tks)


In [4]:
# 2. Branch definitions (exactly four; deterministic names) and statistical granularity.
EXPECTED_BRANCH_NAMES = ["exp_A_gsn_minmax", "exp_B_gsn_zscore", "exp_C_tbsn_minmax", "exp_D_tbsn_zscore"]
SEQ, CHAN = "sequence-wise", "channel-wise"
# AUTHORITATIVE EXPERIMENT DESIGN:  GSN -> sequence-wise,  TBSN -> channel-wise
_BRANCH_TEMPLATE = [
    ("Exp_A", "exp_A_gsn_minmax",  "GSN",  "Min-Max", SEQ),
    ("Exp_B", "exp_B_gsn_zscore",  "GSN",  "Z-Score", SEQ),
    ("Exp_C", "exp_C_tbsn_minmax", "TBSN", "Min-Max", CHAN),
    ("Exp_D", "exp_D_tbsn_zscore", "TBSN", "Z-Score", CHAN),
]
BRANCH_DEFINITION_ERRORS: List[str] = []
BRANCHES = [
    {
        "experiment": exp, "branch_name": name, "source_method": src, "scaling_method": method,
        "scaling_code": "minmax" if method == "Min-Max" else "zscore", "granularity": gran,
        "stats_definition": (
            "finite values pooled over all frames of all eligible trials of the source (report-reproduction scope)"
            if gran == CHAN else "finite values of each individual 751-frame walking trial, per feature"),
    }
    for exp, name, src, method, gran in _BRANCH_TEMPLATE
]


def scaler_filename(b) -> str:
    return "channelwise_statistics.csv" if b["granularity"] == "channel-wise" else "sequence_wise_statistics.csv"


def branch_out_dir(name: str) -> Path:
    return CURRENT_PROCESSED_DIR / name


def branch_mask_dir(name: str) -> Path:
    return CURRENT_MASKS_DIR / name


def branch_scaler_dir(name: str) -> Path:
    return CURRENT_SCALERS_DIR / name


names_ = [b["branch_name"] for b in BRANCHES]
if len(set(names_)) != len(names_):
    BRANCH_DEFINITION_ERRORS.append("Duplicate branch names")
if names_ != EXPECTED_BRANCH_NAMES:
    BRANCH_DEFINITION_ERRORS.append(f"Branch names {names_} != expected {EXPECTED_BRANCH_NAMES}")
if {(b["source_method"], b["scaling_code"]) for b in BRANCHES} != {
        ("GSN", "minmax"), ("GSN", "zscore"), ("TBSN", "minmax"), ("TBSN", "zscore")}:
    BRANCH_DEFINITION_ERRORS.append("Branches do not cover {GSN,TBSN} x {Min-Max,Z-score} exactly once")
FOUR_BRANCHES_PRESENT = len(BRANCHES) == 4 and names_ == EXPECTED_BRANCH_NAMES
BRANCH_DEFS_OK = not BRANCH_DEFINITION_ERRORS

for b in BRANCHES:
    for base in [branch_out_dir(b["branch_name"]), branch_mask_dir(b["branch_name"]), branch_scaler_dir(b["branch_name"])]:
        current_path(base).mkdir(parents=True, exist_ok=False)

mapping_df = pd.DataFrame([{k: b[k] for k in
                            ["experiment", "branch_name", "source_method", "scaling_method", "granularity", "stats_definition"]}
                           for b in BRANCHES])
mapping_df.insert(0, "granularity_profile", DESIGN_LABEL)
mapping_df.to_csv(current_path(CURRENT_AUDIT_DIR, "branch_definitions.csv"), index=False)

print(mapping_df.drop(columns=["stats_definition"]).to_string(index=False))
print("\nStatistical granularity:")
for b in BRANCHES:
    print(f"  {b['branch_name']:<20s} {b['granularity']:<14s} -> {b['stats_definition']}")
if BRANCH_DEFINITION_ERRORS:
    print("\nBRANCH DEFINITION ERRORS:", BRANCH_DEFINITION_ERRORS)

                               granularity_profile experiment       branch_name source_method scaling_method   granularity
final_design__GSN_sequence_wise__TBSN_channel_wise      Exp_A  exp_A_gsn_minmax           GSN        Min-Max sequence-wise
final_design__GSN_sequence_wise__TBSN_channel_wise      Exp_B  exp_B_gsn_zscore           GSN        Z-Score sequence-wise
final_design__GSN_sequence_wise__TBSN_channel_wise      Exp_C exp_C_tbsn_minmax          TBSN        Min-Max  channel-wise
final_design__GSN_sequence_wise__TBSN_channel_wise      Exp_D exp_D_tbsn_zscore          TBSN        Z-Score  channel-wise

Statistical granularity:
  exp_A_gsn_minmax     sequence-wise  -> finite values of each individual 751-frame walking trial, per feature
  exp_B_gsn_zscore     sequence-wise  -> finite values of each individual 751-frame walking trial, per feature
  exp_C_tbsn_minmax    channel-wise   -> finite values pooled over all frames of all eligible trials of the source (report-reproduction s

## 3. NaN-aware scaling core
All statistics use finite values only; the transformation is applied to finite values only; invalid positions stay `NaN`; the mask is the finiteness pattern of the source.

In [5]:
# 3. NaN-aware scaling functions (pure NumPy; no imputation anywhere).
F = len(feature_names) if feature_names else EXPECTED_FEATURES


def sequence_stats(X: np.ndarray, M: np.ndarray) -> dict:
    """Per-feature statistics from VALID (finite) values only. Used per trial (sequence-wise)."""
    n = M.sum(axis=0).astype(np.int64)
    Xm = np.where(M, X, np.nan)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)       # all-NaN slices are expected and handled
        xmin = np.nanmin(Xm, axis=0)
        xmax = np.nanmax(Xm, axis=0)
        mean = np.nanmean(Xm, axis=0)
        std = np.nanstd(Xm, axis=0, ddof=0)
    return finalize_stats(n, xmin, xmax, mean, std)


def finalize_stats(n, xmin, xmax, mean, std) -> dict:
    """Deterministic constant-channel rule: constant <=> valid values exist and xmax == xmin."""
    has = n > 0
    const = has & (xmax == xmin)
    return {
        "n_valid": np.asarray(n, dtype=np.int64),
        "xmin": xmin, "xmax": xmax,
        "rng": np.where(has, xmax - xmin, np.nan),
        "mean": np.where(const, xmin, mean),
        "std": np.where(const, 0.0, std),
        "const": const,
        "no_valid": ~has,
    }


class ChannelAccumulator:
    """Streaming channel-wise statistics over many trials (Chan et al. parallel merge), finite values only."""

    def __init__(self, n_features: int):
        self.n = np.zeros(n_features, dtype=np.int64)
        self.mean = np.zeros(n_features)
        self.m2 = np.zeros(n_features)
        self.xmin = np.full(n_features, np.nan)
        self.xmax = np.full(n_features, np.nan)
        self.trials_with_valid = np.zeros(n_features, dtype=np.int64)
        self.fit_trials = 0

    def update(self, X: np.ndarray, M: np.ndarray) -> None:
        b = sequence_stats(X, M)
        nb = b["n_valid"]
        take = nb > 0
        mb = np.where(take, b["mean"], 0.0)
        m2b = np.where(take, nb * np.where(take, b["std"], 0.0) ** 2, 0.0)
        na = self.n
        n = na + nb
        with np.errstate(all="ignore"):
            delta = mb - self.mean
            new_mean = self.mean + delta * nb / n
            new_m2 = self.m2 + m2b + delta ** 2 * na * nb / n
        first = take & (na == 0)
        self.mean = np.where(first, mb, np.where(take, new_mean, self.mean))
        self.m2 = np.where(first, m2b, np.where(take, new_m2, self.m2))
        self.n = n
        self.xmin = np.fmin(self.xmin, b["xmin"])
        self.xmax = np.fmax(self.xmax, b["xmax"])
        self.trials_with_valid = self.trials_with_valid + take.astype(np.int64)
        self.fit_trials += 1

    def finalize(self) -> dict:
        with np.errstate(all="ignore"):
            std = np.where(self.n > 0, np.sqrt(self.m2 / np.where(self.n > 0, self.n, 1)), np.nan)
        S = finalize_stats(self.n, self.xmin, self.xmax, np.where(self.n > 0, self.mean, np.nan), std)
        S["trials_with_valid"] = self.trials_with_valid.copy()
        S["fit_trials"] = int(self.fit_trials)
        return S


def apply_scaling(X: np.ndarray, M: np.ndarray, S: dict, method: str) -> np.ndarray:
    """Scale finite values with statistics S; invalid positions are returned as NaN (never 0)."""
    safe = S["const"] | S["no_valid"]
    with np.errstate(all="ignore"):
        if method == "Min-Max":
            Y = (X - S["xmin"]) / np.where(safe, 1.0, S["rng"])
        elif method == "Z-Score":
            Y = (X - S["mean"]) / np.where(safe, 1.0, S["std"])
        else:
            raise ValueError(f"Unknown scaling method {method}")
    Y = np.where(S["const"][None, :], 0.0, Y)     # constant channel: valid values -> 0
    return np.where(M, Y, np.nan)                  # missing/invalid stay NaN


def stats_to_frame(S: dict, method: str, feats: List[str]) -> pd.DataFrame:
    d = {"feature": feats, "n_valid": S["n_valid"]}
    if method == "Min-Max":
        d.update({"X_min": S["xmin"], "X_max": S["xmax"], "range": S["rng"], "constant_flag": S["const"]})
    else:
        d.update({"mean": S["mean"], "std": S["std"], "zero_variance_flag": S["const"]})
    d["no_valid_values_flag"] = S["no_valid"]
    return pd.DataFrame(d)


def stats_from_frame(df: pd.DataFrame, method: str) -> dict:
    nan = np.full(len(df), np.nan)
    S = {"n_valid": df["n_valid"].to_numpy(dtype=np.int64),
         "no_valid": df["no_valid_values_flag"].to_numpy(dtype=bool),
         "xmin": nan, "xmax": nan, "rng": nan, "mean": nan, "std": nan}
    if method == "Min-Max":
        S.update(xmin=df["X_min"].to_numpy(float), xmax=df["X_max"].to_numpy(float),
                 rng=df["range"].to_numpy(float), const=df["constant_flag"].to_numpy(dtype=bool))
    else:
        S.update(mean=df["mean"].to_numpy(float), std=df["std"].to_numpy(float),
                 const=df["zero_variance_flag"].to_numpy(dtype=bool))
    return S


def independent_column_stats(X: np.ndarray):
    """Deliberately different implementation (per-column loop) used ONLY by validation."""
    f = X.shape[1]
    n = np.zeros(f, dtype=np.int64)
    mn, mx, mean, std = (np.full(f, np.nan) for _ in range(4))
    for j in range(f):
        v = X[:, j]
        v = v[np.isfinite(v)]
        if v.size:
            n[j], mn[j], mx[j], mean[j], std[j] = v.size, v.min(), v.max(), v.mean(), v.std()
    return n, mn, mx, mean, std


def read_source_trial(source: str, tk: str) -> dict:
    path = SOURCE_DIRS[source] / f"{tk}.csv"
    if not path.is_file():
        raise FileNotFoundError(f"{source} trial file missing: {path}")
    df = pd.read_csv(path)
    cols = df.columns.tolist()
    if cols[:2] != META_COLUMNS:
        raise ValueError(f"first columns must be exactly {META_COLUMNS}, found {cols[:2]}")
    if len(df) != EXPECTED_FRAMES:
        raise ValueError(f"expected {EXPECTED_FRAMES} frames, found {len(df)}")
    body = cols[2:]
    missing = [c for c in feature_names if c not in body]
    extra = [c for c in body if c not in set(feature_names)]
    if missing or extra or len(body) != len(feature_names):
        raise ValueError(f"sensor columns mismatch: missing={missing[:3]} extra={extra[:3]} n={len(body)}")
    frame, time_ = df["frame"].to_numpy(), df["time"].to_numpy()
    if not (np.isfinite(frame.astype(float)).all() and np.isfinite(time_.astype(float)).all()):
        raise ValueError("frame/time metadata contains non-finite values")
    return {"X": df[feature_names].to_numpy(dtype=np.float64), "frame": frame, "time": time_,
            "path": path}


print("Scaling core defined.")

Scaling core defined.


In [6]:
# 4. Unit tests of the scaling core (executed; their result is a mandatory gate).
def run_unit_tests() -> Dict[str, bool]:
    R: Dict[str, bool] = {}
    rng = np.random.default_rng(0)
    with warnings.catch_warnings():
        warnings.simplefilter("error")        # any leaked RuntimeWarning (div-by-zero, all-NaN) fails the test

        # T1 conceptual example [1, 2, NaN, 4]
        X = np.array([[1.0], [2.0], [np.nan], [4.0]]); M = np.isfinite(X)
        S = sequence_stats(X, M)
        Y = apply_scaling(X, M, S, "Min-Max")
        R["minmax_values_and_nan_preserved"] = bool(
            np.isnan(Y[2, 0]) and np.allclose(Y[[0, 1, 3], 0], [0.0, 1 / 3, 1.0]))
        Y = apply_scaling(X, M, S, "Z-Score")
        v = Y[M[:, 0], 0]
        R["zscore_moments_and_nan_preserved"] = bool(
            np.isnan(Y[2, 0]) and abs(v.mean()) < 1e-12 and abs(v.std() - 1) < 1e-12)

        # T2 missing is never filled with zero
        R["nan_not_zero_filled"] = bool(np.isnan(apply_scaling(X, M, S, "Min-Max")[2, 0])
                                        and not (apply_scaling(X, M, S, "Min-Max")[2, 0] == 0))

        # T3 constant channel -> 0 (valid only), NaN kept, no Inf; includes a float value that is not exactly representable
        X = np.array([[0.1, 5.0], [0.1, np.nan], [np.nan, 5.0], [0.1, 5.0]] * 5); M = np.isfinite(X)
        S = sequence_stats(X, M)
        ok = True
        for method in ["Min-Max", "Z-Score"]:
            Y = apply_scaling(X, M, S, method)
            ok &= bool(np.all(Y[M] == 0.0) and np.array_equal(np.isnan(Y), ~M) and not np.isinf(Y).any())
        R["constant_channel_maps_to_zero_safely"] = bool(ok and S["const"].all())

        # T4 feature with no valid value stays all-NaN; single valid value is constant
        X = np.array([[np.nan, 3.0], [np.nan, np.nan], [np.nan, np.nan]]); M = np.isfinite(X)
        S = sequence_stats(X, M)
        Y = apply_scaling(X, M, S, "Z-Score")
        R["all_nan_channel_stays_nan"] = bool(S["no_valid"][0] and np.isnan(Y[:, 0]).all()
                                              and S["const"][1] and Y[0, 1] == 0.0)

        # T5 +/-Inf is invalid -> NaN, mask 0, statistics unaffected
        X = np.array([[1.0], [np.inf], [3.0], [-np.inf]]); M = np.isfinite(X)
        S = sequence_stats(X, M)
        Y = apply_scaling(X, M, S, "Min-Max")
        R["inf_treated_as_invalid"] = bool(np.array_equal(np.isnan(Y[:, 0]), [False, True, False, True])
                                           and np.allclose(Y[[0, 2], 0], [0.0, 1.0]) and not np.isinf(Y).any())

        # T6 streaming channel statistics == direct pooled statistics (finite only)
        trials = []
        for _ in range(6):
            A = rng.normal(5, 3, size=(40, 4)); A[rng.random(A.shape) < 0.2] = np.nan
            A[:, 3] = np.nan if len(trials) < 3 else A[:, 3]          # feature 3: valid only in later trials
            A[:, 2] = 7.0                                              # feature 2: constant
            trials.append(A)
        acc = ChannelAccumulator(4)
        for A in trials:
            acc.update(A, np.isfinite(A))
        S = acc.finalize()
        P = np.vstack(trials)
        n, mn, mx, mean, std = independent_column_stats(P)
        R["channel_accumulator_matches_pooled"] = bool(
            np.array_equal(S["n_valid"], n)
            and np.allclose(S["xmin"], mn, equal_nan=True) and np.allclose(S["xmax"], mx, equal_nan=True)
            and np.allclose(S["mean"], mean, rtol=1e-12, atol=1e-12, equal_nan=True)
            and np.allclose(S["std"], std, rtol=1e-10, atol=1e-12, equal_nan=True)
            and S["const"][2])

        # T7 mask is exactly the finiteness pattern and unchanged by scaling
        A = trials[0]; M = np.isfinite(A)
        Y = apply_scaling(A, M, S, "Z-Score")
        R["mask_aligned_after_scaling"] = bool(np.array_equal(np.isfinite(Y), M))
    return R


UNIT_TEST_RESULTS = {}
def _stage_unit_tests():
    UNIT_TEST_RESULTS.update(run_unit_tests())

run_stage("unit_tests", _stage_unit_tests)
UNIT_TESTS_PASS = bool(UNIT_TEST_RESULTS) and all(UNIT_TEST_RESULTS.values())
for k, v in UNIT_TEST_RESULTS.items():
    print(f"{'PASS' if v else 'FAIL'}  {k}")
print("UNIT TESTS:", "PASS" if UNIT_TESTS_PASS else "FAIL")

PASS  minmax_values_and_nan_preserved
PASS  zscore_moments_and_nan_preserved
PASS  nan_not_zero_filled
PASS  constant_channel_maps_to_zero_safely
PASS  all_nan_channel_stays_nan
PASS  inf_treated_as_invalid
PASS  channel_accumulator_matches_pooled
PASS  mask_aligned_after_scaling
UNIT TESTS: PASS


## 5. Input inventory and missingness audit (pass 1)
Every trial of each source is read once: schema is validated, the missingness pattern is audited, eligibility is decided (excluded only if it has **no** finite sensor value), and channel-wise statistics are accumulated from finite values of eligible trials. Nothing is imputed.

In [7]:
# 5. Pass 1 — inventory, missingness audit, eligibility, channel-wise statistics.
INVENTORY: Dict[tuple, dict] = {}
CHANNEL_STATS: Dict[str, dict] = {}


def stage_inventory():
    for source in SOURCES:
        needs_channel = any(b["source_method"] == source and b["granularity"] == "channel-wise" for b in BRANCHES)
        acc = ChannelAccumulator(len(feature_names)) if needs_channel else None
        for tk in common_keys:
            path = SOURCE_DIRS[source] / f"{tk}.csv"
            base = {"source": source, "trial_key": tk, "source_file": rel(path)}
            try:
                src = read_source_trial(source, tk)
                X = src["X"]
                fin = np.isfinite(X)
                n_valid = int(fin.sum())
                row = {**base,
                       "n_valid": n_valid, "n_missing": int(X.size - n_valid),
                       "missing_fraction": float((X.size - n_valid) / X.size),
                       "n_inf": int(np.isinf(X).sum()),
                       "n_features_fully_missing": int((~fin).all(axis=0).sum()),
                       "n_features_with_any_missing": int((~fin).any(axis=0).sum()),
                       "status": "ELIGIBLE", "stage": "", "reason": ""}
                if n_valid < MIN_FINITE_VALUES_PER_TRIAL:
                    row.update(status="EXCLUDED", stage="input_validity_check",
                               reason=(f"Only {n_valid} finite sensor values in the {EXPECTED_FRAMES}x{len(feature_names)} "
                                       f"matrix (< {MIN_FINITE_VALUES_PER_TRIAL}); nothing to scale. No imputation performed."))
                elif acc is not None and SPLIT_MAP.get(tk) == "TRAIN":
                    acc.update(X, fin)
            except Exception as exc:
                row = {**base, "n_valid": np.nan, "n_missing": np.nan, "missing_fraction": np.nan, "n_inf": np.nan,
                       "n_features_fully_missing": np.nan, "n_features_with_any_missing": np.nan,
                       "status": "FAILED", "stage": "input_load_and_schema", "reason": repr(exc)}
            INVENTORY[(source, tk)] = row
        if acc is not None:
            CHANNEL_STATS[source] = acc.finalize()


if INPUT_OK and BRANCH_DEFS_OK:
    run_stage("pass1_inventory", stage_inventory)
else:
    print("Skipping pass 1: input contract or branch definitions invalid.")

INPUT_MISSINGNESS_DF = pd.DataFrame(list(INVENTORY.values()))
if len(INPUT_MISSINGNESS_DF):
    INPUT_MISSINGNESS_DF.insert(0, "run_id", RUN_ID)
INPUT_MISSINGNESS_DF.to_csv(current_path(CURRENT_AUDIT_DIR, "input_missingness_audit.csv"), index=False)

print("=== INPUT INVENTORY / MISSINGNESS ===")
if len(INPUT_MISSINGNESS_DF):
    for source in SOURCES:
        d = INPUT_MISSINGNESS_DF[INPUT_MISSINGNESS_DF.source == source]
        ok = d[d.status != "FAILED"]
        print(f"{source}: trials={len(d)} eligible={int((d.status=='ELIGIBLE').sum())} "
              f"excluded={int((d.status=='EXCLUDED').sum())} failed={int((d.status=='FAILED').sum())} | "
              f"trials with >=1 missing value={int((ok.n_missing > 0).sum())} | "
              f"trials with Inf={int((ok.n_inf > 0).sum())} | "
              f"mean missing fraction={ok.missing_fraction.mean():.4f}")


=== INPUT INVENTORY / MISSINGNESS ===
GSN: trials=881 eligible=881 excluded=0 failed=0 | trials with >=1 missing value=0 | trials with Inf=0 | mean missing fraction=0.0000
TBSN: trials=881 eligible=881 excluded=0 failed=0 | trials with >=1 missing value=0 | trials with Inf=0 | mean missing fraction=0.0000


## 6. Scale and write the four branches (pass 2)
Fresh run directories only. Each trial is written as `frame, time, <180 sensor features>` (NaN preserved) plus a `uint8` validity mask `.npy` of shape 751 × 180. Scalers and per-branch metadata go to `CURRENT_SCALERS_DIR`.

In [8]:
# 6. Pass 2 — scaling, writing, provenance records.
TRIAL_RECORDS: List[dict] = []
CONSTANT_RECORDS: List[dict] = []
SEQ_STATS_BLOCKS: Dict[str, List[pd.DataFrame]] = {}


def make_record(b: dict, tk: str, inv: dict, status: str, stage: str = "", reason: str = "",
                out_path=None, mask_path=None) -> dict:
    meta = TRIAL_META_BY_KEY.get(tk, {})
    scaler_path = branch_scaler_dir(b["branch_name"]) / scaler_filename(b)
    ok_stats = status in ("COMPLETED", "EXCLUDED")
    return {
        "run_id": RUN_ID, "timestamp_utc": RUN_TIMESTAMP,
        "branch_name": b["branch_name"], "experiment": b["experiment"],
        "source_method": b["source_method"], "scaling_method": b["scaling_method"],
        "scaling_granularity": b["granularity"], "granularity_profile": DESIGN_LABEL,
        "trial_key": tk, "subject_id": meta.get("subject_id", "NA"), "condition": meta.get("condition", "NA"),
        "walk": meta.get("walk", "NA"), "trial": meta.get("trial", "NA"),
        "status": status,
        "exclusion_stage": stage if status == "EXCLUDED" else "",
        "exclusion_reason": reason if status == "EXCLUDED" else "",
        "failure_stage": stage if status == "FAILED" else "",
        "failure_reason": reason if status == "FAILED" else "",
        "number_of_valid_values": inv.get("n_valid", np.nan) if ok_stats else np.nan,
        "number_of_missing_values": inv.get("n_missing", np.nan) if ok_stats else np.nan,
        "missing_fraction": inv.get("missing_fraction", np.nan) if ok_stats else np.nan,
        "n_frames": EXPECTED_FRAMES, "n_features": len(feature_names), "feature_hash": FEATURE_HASH,
        "source_file": inv.get("source_file", ""),
        "output_file": rel(out_path) if out_path is not None else "",
        "mask_file": rel(mask_path) if mask_path is not None else "",
        "scaler_file": rel(scaler_path) if status == "COMPLETED" else "",
    }


def stage_scale_and_write():
    # -- channel-wise scaler files first (statistics are final after pass 1) --
    for b in BRANCHES:
        name = b["branch_name"]
        SEQ_STATS_BLOCKS[name] = []
        if b["granularity"] == "channel-wise" and b["source_method"] in CHANNEL_STATS:
            S = CHANNEL_STATS[b["source_method"]]
            sf = stats_to_frame(S, b["scaling_method"], feature_names)
            sf.insert(0, "scaling_method", b["scaling_method"])
            sf.insert(0, "source_method", b["source_method"])
            sf["fit_trial_count"] = S["trials_with_valid"]
            sf["fit_observation_count"] = S["n_valid"]
            sf["run_id"] = RUN_ID
            sf.to_csv(current_path(branch_scaler_dir(name), scaler_filename(b)), index=False)
            flag = "constant_flag" if b["scaling_method"] == "Min-Max" else "zero_variance_flag"
            for _, r in sf[sf[flag] | sf["no_valid_values_flag"]].iterrows():
                CONSTANT_RECORDS.append({"run_id": RUN_ID, "branch_name": name, "scope": "channel-wise",
                                         "trial_key": "ALL_ELIGIBLE", "feature": r["feature"],
                                         "condition": "no_valid_values" if r["no_valid_values_flag"] else flag})

    for source in SOURCES:
        branches_here = [b for b in BRANCHES if b["source_method"] == source]
        for tk in common_keys:
            inv = INVENTORY[(source, tk)]
            if inv["status"] != "ELIGIBLE":
                for b in branches_here:
                    TRIAL_RECORDS.append(make_record(b, tk, inv, inv["status"], inv["stage"], inv["reason"]))
                continue
            try:
                src = read_source_trial(source, tk)
                X = src["X"]
                M = np.isfinite(X)
            except Exception as exc:
                for b in branches_here:
                    TRIAL_RECORDS.append(make_record(b, tk, inv, "FAILED", "input_load_and_schema", repr(exc)))
                continue
            for b in branches_here:
                name = b["branch_name"]
                out_path = current_path(branch_out_dir(name), f"{tk}.csv")
                mask_path = current_path(branch_mask_dir(name), f"{tk}.npy")
                try:
                    if b["granularity"] == "channel-wise":
                        S = CHANNEL_STATS[source]
                    else:
                        S = sequence_stats(X, M)
                        sf = stats_to_frame(S, b["scaling_method"], feature_names)
                        sf.insert(0, "scaling_method", b["scaling_method"])
                        sf.insert(0, "experiment", b["experiment"])
                        sf.insert(0, "trial_key", tk)
                        SEQ_STATS_BLOCKS[name].append(sf)
                        flag = "constant_flag" if b["scaling_method"] == "Min-Max" else "zero_variance_flag"
                        for _, r in sf[sf[flag] | sf["no_valid_values_flag"]].iterrows():
                            CONSTANT_RECORDS.append({"run_id": RUN_ID, "branch_name": name, "scope": "sequence-wise",
                                                     "trial_key": tk, "feature": r["feature"],
                                                     "condition": "no_valid_values" if r["no_valid_values_flag"] else flag})
                    Y = apply_scaling(X, M, S, b["scaling_method"])
                    assert Y.shape == (EXPECTED_FRAMES, len(feature_names)), f"bad output shape {Y.shape}"
                    assert np.array_equal(np.isnan(Y), ~M), "NaN pattern differs from validity mask"
                    out = pd.DataFrame(Y, columns=feature_names)
                    out.insert(0, "time", src["time"])
                    out.insert(0, "frame", src["frame"])
                    out.to_csv(out_path, index=False)
                    np.save(mask_path, M.astype(np.uint8))
                    TRIAL_RECORDS.append(make_record(b, tk, inv, "COMPLETED", out_path=out_path, mask_path=mask_path))
                except Exception as exc:
                    for p in (out_path, mask_path):
                        if p.exists():
                            p.unlink()
                    TRIAL_RECORDS.append(make_record(b, tk, inv, "FAILED", "scaling_or_write", repr(exc)))

    # -- sequence-wise scaler files + per-branch metadata --
    for b in BRANCHES:
        name = b["branch_name"]
        if b["granularity"] == "sequence-wise" and SEQ_STATS_BLOCKS[name]:
            sf = pd.concat(SEQ_STATS_BLOCKS[name], ignore_index=True)
            sf["run_id"] = RUN_ID
            sf.to_csv(current_path(branch_scaler_dir(name), scaler_filename(b)), index=False)


if INPUT_OK and BRANCH_DEFS_OK and "pass1_inventory" not in STAGE_ERRORS:
    run_stage("pass2_scale_and_write", stage_scale_and_write)
else:
    print("Skipping pass 2: earlier stage invalid.")

TRIAL_RECORDS_DF = pd.DataFrame(TRIAL_RECORDS)


def write_branch_metadata():
    for b in BRANCHES:
        d = TRIAL_RECORDS_DF[TRIAL_RECORDS_DF.branch_name == b["branch_name"]] if len(TRIAL_RECORDS_DF) else TRIAL_RECORDS_DF
        meta = {k: b[k] for k in ["experiment", "branch_name", "source_method", "scaling_method", "granularity", "stats_definition"]}
        meta.update({
            "granularity_profile": DESIGN_LABEL, "run_id": RUN_ID, "timestamp_utc": RUN_TIMESTAMP,
            "expected_trials": len(common_keys),
            "completed_trials": int((d.status == "COMPLETED").sum()) if len(d) else 0,
            "excluded_trials": int((d.status == "EXCLUDED").sum()) if len(d) else 0,
            "failed_trials": int((d.status == "FAILED").sum()) if len(d) else 0,
            "n_features": len(feature_names), "n_frames": EXPECTED_FRAMES, "feature_hash": FEATURE_HASH,
            "mask_convention": "M=1 finite observation, M=0 missing/invalid; shape 751x180",
            "imputation_performed": False, "train_validation_test_split_created": False,
        })
        with open(current_path(branch_scaler_dir(b["branch_name"]), "branch_metadata.json"), "w", encoding="utf-8") as f:
            json.dump(meta, f, indent=2)


run_stage("write_branch_metadata", write_branch_metadata)

# -- run-level audit tables (all inside the CURRENT run directory) --
rec_cols = list(TRIAL_RECORDS_DF.columns)
TRIAL_RECORDS_DF.to_csv(current_path(CURRENT_AUDIT_DIR, "scaled_trial_manifest.csv"), index=False)

EXCLUSION_COLUMNS = ["trial_key", "subject_id", "condition", "walk", "trial", "preprocessing_branch",
                     "exclusion_stage", "exclusion_reason", "number_of_valid_values",
                     "number_of_missing_values", "missing_fraction", "run_id", "timestamp_utc"]
if len(TRIAL_RECORDS_DF):
    ex = TRIAL_RECORDS_DF[TRIAL_RECORDS_DF.status == "EXCLUDED"].rename(columns={"branch_name": "preprocessing_branch"})
    EXCLUSIONS_DF = ex[EXCLUSION_COLUMNS].reset_index(drop=True)
    FAILURES_DF = TRIAL_RECORDS_DF[TRIAL_RECORDS_DF.status == "FAILED"][
        ["trial_key", "subject_id", "branch_name", "failure_stage", "failure_reason", "run_id", "timestamp_utc"]
    ].reset_index(drop=True)
else:
    EXCLUSIONS_DF = pd.DataFrame(columns=EXCLUSION_COLUMNS)
    FAILURES_DF = pd.DataFrame(columns=["trial_key", "subject_id", "branch_name", "failure_stage", "failure_reason",
                                        "run_id", "timestamp_utc"])
EXCLUSIONS_DF.to_csv(current_path(CURRENT_AUDIT_DIR, "excluded_trials.csv"), index=False)
FAILURES_DF.to_csv(current_path(CURRENT_AUDIT_DIR, "failed_trials.csv"), index=False)
pd.DataFrame(CONSTANT_RECORDS, columns=["run_id", "branch_name", "scope", "trial_key", "feature", "condition"]).to_csv(
    current_path(CURRENT_AUDIT_DIR, "constant_or_zero_variance_features.csv"), index=False)
if len(trial_meta):
    gm = gsn_manifest.assign(trial_key=gsn_manifest["trial_key"].astype(str))
    gm[gm.trial_key.isin(common_keys)].sort_values("trial_key").to_csv(
        current_path(CURRENT_AUDIT_DIR, "trial_metadata.csv"), index=False)

print("=== PASS 2 SUMMARY (current run) ===")
if len(TRIAL_RECORDS_DF):
    print(TRIAL_RECORDS_DF.groupby(["branch_name", "status"]).size().unstack(fill_value=0).to_string())
print(f"\nExcluded trial rows: {len(EXCLUSIONS_DF)} | Failed trial rows: {len(FAILURES_DF)}")
print(f"Exclusion table : {rel(CURRENT_AUDIT_DIR / 'excluded_trials.csv')}")
if len(EXCLUSIONS_DF):
    print(EXCLUSIONS_DF[["trial_key", "preprocessing_branch", "exclusion_stage", "number_of_valid_values"]].head(20).to_string(index=False))

=== PASS 2 SUMMARY (current run) ===
status             COMPLETED
branch_name                 
exp_A_gsn_minmax         881
exp_B_gsn_zscore         881
exp_C_tbsn_minmax        881
exp_D_tbsn_zscore        881

Excluded trial rows: 0 | Failed trial rows: 0
Exclusion table : data/processed/scaled_features_run_20261009_142420_27f01a/_audit/excluded_trials.csv


## 7. Validation of the CURRENT run (reads outputs and scalers from disk)
Every output, mask and scaler is re-read **from the current run directories only** (path-guarded) and checked against the **source** files re-read independently: structure, frame/time, mask alignment, NaN alignment, scaler consistency (independent recomputation), formula, and mathematical properties (Min-Max range, Z-score mean/std, constant channels). NaN positions are never tested as finite values.

In [9]:
# 7. Exhaustive validation — current-run artifacts only.
VALIDATION_ROWS: List[dict] = []


def validate_branch(b: dict) -> dict:
    name, source, method, gran = b["branch_name"], b["source_method"], b["scaling_method"], b["granularity"]
    recs = TRIAL_RECORDS_DF[TRIAL_RECORDS_DF.branch_name == name] if len(TRIAL_RECORDS_DF) else TRIAL_RECORDS_DF
    completed = sorted(recs[recs.status == "COMPLETED"].trial_key.astype(str).tolist())
    counts = {k: 0 for k in ["missing_output_files", "unexpected_output_files", "missing_mask_files",
                             "unexpected_mask_files", "column_errors", "frame_count_errors", "frame_time_errors",
                             "mask_errors", "nan_alignment_errors", "inf_errors", "scaler_errors",
                             "formula_errors", "math_errors", "exception_errors"]}
    firsts: List[str] = []

    def err(kind: str, tk: str, msg: str = "") -> None:
        counts[kind] += 1
        if len(firsts) < 8:
            firsts.append(f"{kind}[{tk}] {msg}".strip())

    out_dir = current_path(branch_out_dir(name))
    mask_dir = current_path(branch_mask_dir(name))
    scaler_dir = current_path(branch_scaler_dir(name))
    scaler_path = current_path(scaler_dir, scaler_filename(b))
    meta_path = current_path(scaler_dir, "branch_metadata.json")

    # ---- files on disk vs. records (no stale / stray files in a fresh run dir) ----
    on_disk = sorted(p.stem for p in out_dir.glob("*.csv")) if out_dir.is_dir() else []
    mask_disk = sorted(p.stem for p in mask_dir.glob("*.npy")) if mask_dir.is_dir() else []
    counts["missing_output_files"] = len(set(completed) - set(on_disk))
    counts["unexpected_output_files"] = len(set(on_disk) - set(completed))
    counts["missing_mask_files"] = len(set(completed) - set(mask_disk))
    counts["unexpected_mask_files"] = len(set(mask_disk) - set(completed))
    scaler_file_exists = scaler_path.is_file() and meta_path.is_file()

    # ---- load scaler from disk ----
    S_ch, S_seq, scaler_loaded = None, {}, False
    try:
        sdf = pd.read_csv(scaler_path)
        if gran == "channel-wise":
            if sdf["feature"].astype(str).tolist() != feature_names:
                raise ValueError("scaler feature order differs from provenance feature list")
            S_ch = stats_from_frame(sdf, method)
        else:
            if set(sdf["trial_key"].astype(str)) != set(completed):
                raise ValueError("scaler trial coverage differs from completed trials")
            for tk_, g_ in sdf.groupby(sdf["trial_key"].astype(str), sort=False):
                if g_["feature"].astype(str).tolist() != feature_names:
                    raise ValueError(f"scaler feature order differs for {tk_}")
                S_seq[tk_] = stats_from_frame(g_.reset_index(drop=True), method)
        scaler_loaded = True
    except Exception as exc:
        err("scaler_errors", "-", f"cannot load scaler: {exc!r}")

    # ---- channel-wise: independent recomputation of pooled statistics (pass V1) ----
    ref_ch = None
    if gran == "channel-wise" and completed and scaler_loaded:
        n_ = np.zeros(len(feature_names), dtype=np.int64)
        s_ = np.zeros(len(feature_names))
        mn_ = np.full(len(feature_names), np.nan)
        mx_ = np.full(len(feature_names), np.nan)
        for tk in completed:
            if SPLIT_MAP.get(tk) != "TRAIN": continue
            try:
                Xs = read_source_trial(source, tk)["X"]
            except Exception as exc:
                err("scaler_errors", tk, f"source unreadable: {exc!r}")
                continue
            for j in range(Xs.shape[1]):
                v = Xs[:, j]
                v = v[np.isfinite(v)]
                if v.size:
                    n_[j] += v.size
                    s_[j] += v.sum()
                    mn_[j] = np.fmin(mn_[j], v.min())
                    mx_[j] = np.fmax(mx_[j], v.max())
        with np.errstate(all="ignore"):
            mean_ = np.where(n_ > 0, s_ / np.where(n_ > 0, n_, 1), np.nan)
        ref_ch = {"n": n_, "mn": mn_, "mx": mx_, "mean": mean_, "ssd": np.zeros(len(feature_names))}
        ref_ch["const"] = (n_ > 0) & (mn_ == mx_)

    # aggregates over outputs (channel-wise math)
    agg_min = np.full(len(feature_names), np.nan)
    agg_max = np.full(len(feature_names), np.nan)
    agg_n = np.zeros(len(feature_names))
    agg_s = np.zeros(len(feature_names))
    agg_ss = np.zeros(len(feature_names))
    first_cols = META_COLUMNS + feature_names

    for tk in completed:
        try:
            src = read_source_trial(source, tk)
            Xs = src["X"]
            M_exp = np.isfinite(Xs)
            out_path = current_path(out_dir, f"{tk}.csv")
            odf = pd.read_csv(out_path)

            # structure
            if len(odf) != EXPECTED_FRAMES:
                err("frame_count_errors", tk, f"rows={len(odf)}")
            if odf.columns.tolist() != first_cols:
                err("column_errors", tk, f"n_cols={odf.shape[1]}")
                continue
            if not (np.array_equal(odf["frame"].to_numpy(), src["frame"]) and
                    np.array_equal(odf["time"].to_numpy(), src["time"])):
                err("frame_time_errors", tk)
            if len(odf) != EXPECTED_FRAMES:
                continue
            Y = odf[feature_names].to_numpy(dtype=np.float64)

            # mask
            mpath = current_path(mask_dir, f"{tk}.npy")
            Msv = np.load(mpath)
            if Msv.shape != (EXPECTED_FRAMES, len(feature_names)) or Msv.dtype != np.uint8 or not np.isin(Msv, [0, 1]).all():
                err("mask_errors", tk, f"shape={Msv.shape} dtype={Msv.dtype}")
                continue
            Mb = Msv.astype(bool)
            if not np.array_equal(Mb, M_exp):
                err("mask_errors", tk, "saved mask differs from finiteness of the source")
            if not np.array_equal(np.isnan(Y), ~Mb):
                err("nan_alignment_errors", tk, "NaN positions in X differ from invalid mask entries")
            if not np.array_equal(np.isfinite(Y), Mb):
                err("nan_alignment_errors", tk, "valid mask entries are not finite")
            if np.isinf(Y).any():
                err("inf_errors", tk)

            # scaler statistics for this trial
            if not scaler_loaded:
                continue
            if gran == "sequence-wise":
                S = S_seq.get(tk)
                if S is None:
                    err("scaler_errors", tk, "no scaler rows for trial")
                    continue
                rn, rmn, rmx, rmean, rstd = independent_column_stats(Xs)
                rconst = (rn > 0) & (rmx == rmn)
                ok = (np.array_equal(S["n_valid"], rn) and np.array_equal(S["const"], rconst)
                      and np.array_equal(S["no_valid"], rn == 0))
                if method == "Min-Max":
                    ok = ok and np.allclose(S["xmin"], rmn, rtol=STATS_RTOL, atol=STATS_ATOL, equal_nan=True) \
                            and np.allclose(S["xmax"], rmx, rtol=STATS_RTOL, atol=STATS_ATOL, equal_nan=True)
                else:
                    ok = ok and np.allclose(S["mean"], rmean, rtol=STATS_RTOL, atol=STATS_ATOL, equal_nan=True) \
                            and np.allclose(S["std"], rstd, rtol=STATS_RTOL, atol=STATS_ATOL, equal_nan=True)
                if not ok:
                    err("scaler_errors", tk, "stored sequence-wise statistics differ from independent recomputation")
                const_here = rconst
                nonconst_here = (rn > 0) & ~rconst
            else:
                S = S_ch
                const_here = ref_ch["const"] if ref_ch else S["const"]
                nonconst_here = (ref_ch["n"] > 0) & ~ref_ch["const"] if ref_ch else ~S["const"]
                if SPLIT_MAP.get(tk) == "TRAIN":
                    ref_ch["ssd"] += np.array([
                        np.sum((Xs[np.isfinite(Xs[:, j]), j] - ref_ch["mean"][j]) ** 2) if ref_ch["n"][j] > 0 else 0.0
                        for j in range(len(feature_names))])

            # formula: output == formula(source, STORED scaler)
            expected = apply_scaling(Xs, M_exp, S, method)
            if not np.allclose(Y[M_exp], expected[M_exp], rtol=FORMULA_RTOL, atol=FORMULA_ATOL):
                err("formula_errors", tk, "output differs from formula(source, stored scaler)")

            # mathematics (finite positions only)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                ymin, ymax = np.nanmin(Y, axis=0), np.nanmax(Y, axis=0)
                ymean, ystd = np.nanmean(Y, axis=0), np.nanstd(Y, axis=0, ddof=0)
            if const_here.any() and not np.all(Y[:, const_here][M_exp[:, const_here]] == 0.0):
                err("math_errors", tk, "constant channel not mapped to 0")
            if method == "Min-Max":
                if np.isfinite(Y).any() and (np.nanmin(Y) < -RANGE_TOL or np.nanmax(Y) > 1 + RANGE_TOL):
                    if gran == "sequence-wise" or SPLIT_MAP.get(tk) == "TRAIN":
                        err("math_errors", tk, "finite values outside [0,1]")
                if gran == "sequence-wise":
                    bad = nonconst_here & ((np.abs(ymin) > RANGE_TOL) | (np.abs(ymax - 1) > RANGE_TOL))
                    if bad.any():
                        err("math_errors", tk, f"{int(bad.sum())} features do not span exactly [0,1]")
            else:
                if gran == "sequence-wise":
                    bad = nonconst_here & ((np.abs(ymean) > MOMENT_TOL) | (np.abs(ystd - 1) > MOMENT_TOL))
                    if bad.any():
                        err("math_errors", tk, f"{int(bad.sum())} features violate mean=0/std=1")
            if gran == "channel-wise" and SPLIT_MAP.get(tk) == "TRAIN":
                agg_min, agg_max = np.fmin(agg_min, ymin), np.fmax(agg_max, ymax)
                agg_n += M_exp.sum(axis=0)
                agg_s += np.where(M_exp, Y, 0.0).sum(axis=0)
                agg_ss += np.square(np.where(M_exp, Y, 0.0)).sum(axis=0)
        except Exception as exc:
            err("exception_errors", tk, repr(exc))

    # ---- channel-wise aggregate checks ----
    if gran == "channel-wise" and completed and scaler_loaded and ref_ch is not None:
        with np.errstate(all="ignore"):
            rstd = np.where(ref_ch["n"] > 0, np.sqrt(ref_ch["ssd"] / np.where(ref_ch["n"] > 0, ref_ch["n"], 1)), np.nan)
        if not (np.array_equal(S_ch["n_valid"], ref_ch["n"]) and np.array_equal(S_ch["const"], ref_ch["const"])):
            err("scaler_errors", "-", "stored channel counts/constant flags differ from independent recomputation")
        if method == "Min-Max":
            if not (np.allclose(S_ch["xmin"], ref_ch["mn"], rtol=STATS_RTOL, atol=STATS_ATOL, equal_nan=True) and
                    np.allclose(S_ch["xmax"], ref_ch["mx"], rtol=STATS_RTOL, atol=STATS_ATOL, equal_nan=True)):
                err("scaler_errors", "-", "stored channel min/max differ from independent recomputation")
        else:
            if not (np.allclose(S_ch["mean"], ref_ch["mean"], rtol=STATS_RTOL, atol=STATS_ATOL, equal_nan=True) and
                    np.allclose(S_ch["std"], rstd, rtol=STATS_RTOL, atol=STATS_ATOL, equal_nan=True)):
                err("scaler_errors", "-", "stored channel mean/std differ from independent recomputation")
        nonconst = (ref_ch["n"] > 0) & ~ref_ch["const"]
        if method == "Min-Max":
            if (nonconst & ((np.abs(agg_min) > RANGE_TOL) | (np.abs(agg_max - 1) > RANGE_TOL))).any():
                err("math_errors", "-", "channel-wise Min-Max features do not span exactly [0,1] over all trials")
        else:
            with np.errstate(all="ignore"):
                m_ = agg_s / np.where(agg_n > 0, agg_n, 1)
                s_ = np.sqrt(np.maximum(agg_ss / np.where(agg_n > 0, agg_n, 1) - m_ ** 2, 0.0))
            if (nonconst & ((np.abs(m_) > MOMENT_TOL) | (np.abs(s_ - 1) > MOMENT_TOL))).any():
                err("math_errors", "-", "channel-wise Z-score features violate pooled mean=0/std=1")

    has_completed = len(completed) > 0
    res = {
        "branch_name": name, "validated_trials": len(completed), **counts,
        "scaler_files_exist": bool(scaler_file_exists),
        "output_files_ok": bool(has_completed and counts["missing_output_files"] == 0 and counts["unexpected_output_files"] == 0
                                and counts["missing_mask_files"] == 0 and counts["unexpected_mask_files"] == 0),
        "feature_count_ok": bool(has_completed and counts["column_errors"] == 0),
        "frame_count_ok": bool(has_completed and counts["frame_count_errors"] == 0 and counts["frame_time_errors"] == 0),
        "mask_alignment_ok": bool(has_completed and counts["mask_errors"] == 0 and counts["nan_alignment_errors"] == 0
                                  and counts["inf_errors"] == 0),
        "scaler_ok": bool(has_completed and scaler_file_exists and scaler_loaded and counts["scaler_errors"] == 0),
        "math_ok": bool(has_completed and counts["formula_errors"] == 0 and counts["math_errors"] == 0
                        and counts["exception_errors"] == 0),
        "first_errors": " | ".join(firsts),
    }
    res["validation_status"] = "PASS" if all(res[k] for k in
                                             ["output_files_ok", "feature_count_ok", "frame_count_ok",
                                              "mask_alignment_ok", "scaler_ok", "math_ok"]) else "FAIL"
    return res


def stage_validation():
    for b in BRANCHES:
        t0 = time.time()
        VALIDATION_ROWS.append(validate_branch(b))
        print(f"validated {b['branch_name']:<20s} in {time.time() - t0:6.1f}s -> {VALIDATION_ROWS[-1]['validation_status']}")


if INPUT_OK and BRANCH_DEFS_OK and "pass2_scale_and_write" in STAGE_SECONDS and "pass2_scale_and_write" not in STAGE_ERRORS:
    run_stage("validation", stage_validation)
else:
    print("Skipping validation: scaling stage did not run successfully.")

VALIDATION_DF = pd.DataFrame(VALIDATION_ROWS)
VALIDATION_DF.to_csv(current_path(CURRENT_AUDIT_DIR, "validation_results.csv"), index=False)
if len(VALIDATION_DF):
    show = ["branch_name", "validated_trials", "output_files_ok", "feature_count_ok", "frame_count_ok",
            "mask_alignment_ok", "scaler_ok", "math_ok", "validation_status"]
    print("\n=== CURRENT-RUN VALIDATION ===")
    print(VALIDATION_DF[show].to_string(index=False))
    bad_rows = VALIDATION_DF[VALIDATION_DF.first_errors != ""]
    for _, r in bad_rows.iterrows():
        print(f"\n[{r.branch_name}] first errors: {r.first_errors}")



validated exp_A_gsn_minmax     in   83.9s -> PASS
validated exp_B_gsn_zscore     in   89.9s -> PASS
validated exp_C_tbsn_minmax    in  104.8s -> PASS
validated exp_D_tbsn_zscore    in  108.6s -> PASS

=== CURRENT-RUN VALIDATION ===
      branch_name  validated_trials  output_files_ok  feature_count_ok  frame_count_ok  mask_alignment_ok  scaler_ok  math_ok validation_status
 exp_A_gsn_minmax               881             True              True            True               True       True     True              PASS
 exp_B_gsn_zscore               881             True              True            True               True       True     True              PASS
exp_C_tbsn_minmax               881             True              True            True               True       True     True              PASS
exp_D_tbsn_zscore               881             True              True            True               True       True     True              PASS


## 7b. Report alignment validation (`REPORT_ALIGNMENT_DF`)
Compares the **mandatory** design (a literal table independent of the branch configuration) with the **actual** configuration, the scaler artifact written to disk, the content of that artifact, and the validation result. Its outcome is a required final gate.

In [10]:
# 7b. REPORT_ALIGNMENT_DF — created here, before it is used by the final gates.
EXPECTED_DESIGN = pd.DataFrame([
    ("Exp_A", "exp_A_gsn_minmax",  "GSN",  "Min-Max", "sequence-wise", "sequence_wise_statistics.csv"),
    ("Exp_B", "exp_B_gsn_zscore",  "GSN",  "Z-Score", "sequence-wise", "sequence_wise_statistics.csv"),
    ("Exp_C", "exp_C_tbsn_minmax", "TBSN", "Min-Max", "channel-wise",  "channelwise_statistics.csv"),
    ("Exp_D", "exp_D_tbsn_zscore", "TBSN", "Z-Score", "channel-wise",  "channelwise_statistics.csv"),
], columns=["Experiment", "Expected Branch", "Expected Source", "Expected Scaler", "Expected Granularity", "Expected Scaler Artifact"])


def build_report_alignment() -> pd.DataFrame:
    rows = []
    actual = {b["experiment"]: b for b in BRANCHES}
    vmap = {r["branch_name"]: r for r in VALIDATION_ROWS}
    for _, e in EXPECTED_DESIGN.iterrows():
        b = actual.get(e.Experiment)
        a_branch = b["branch_name"] if b else "MISSING"
        a_src = b["source_method"] if b else "MISSING"
        a_sc = b["scaling_method"] if b else "MISSING"
        a_gr = b["granularity"] if b else "MISSING"
        art_path = current_path(branch_scaler_dir(a_branch), e["Expected Scaler Artifact"]) if b else None
        art_exists = bool(art_path and art_path.is_file())
        # artifact content must reveal the granularity: per-trial rows (trial_key column) <=> sequence-wise
        content_ok = False
        if art_exists:
            cols = pd.read_csv(art_path, nrows=1).columns
            content_ok = ("trial_key" in cols) == (e["Expected Granularity"] == "sequence-wise")
        v = vmap.get(a_branch)
        checks = {
            "branch_name": a_branch == e["Expected Branch"],
            "source": a_src == e["Expected Source"],
            "scaler": a_sc == e["Expected Scaler"],
            "granularity": a_gr == e["Expected Granularity"],
            "scaler_artifact": art_exists and content_ok,
            "branch_validated": bool(v and v["validation_status"] == "PASS"),
        }
        rows.append({
            "Experiment": e.Experiment,
            "Expected": f"{e['Expected Source']} + {e['Expected Granularity']} {e['Expected Scaler']}",
            "Actual": f"{a_src} + {a_gr} {a_sc}",
            "Branch Name": a_branch, "Preprocessing Source": a_src, "Scaler": a_sc, "Scaling Granularity": a_gr,
            "Expected Output": f"{e['Expected Branch']}/<trial_key>.csv + {e['Expected Scaler Artifact']}",
            "Failed Checks": ",".join(k for k, ok in checks.items() if not ok),
            "Alignment": "PASS" if all(checks.values()) else "FAIL",
        })
    return pd.DataFrame(rows)


REPORT_ALIGNMENT_DF = build_report_alignment()
all_report_alignments_pass = bool(len(REPORT_ALIGNMENT_DF) == 4 and REPORT_ALIGNMENT_DF["Alignment"].eq("PASS").all())
REPORT_ALIGNMENT_DF.to_csv(current_path(CURRENT_AUDIT_DIR, "report_alignment.csv"), index=False)
print("=== REPORT ALIGNMENT ===")
print(REPORT_ALIGNMENT_DF[["Experiment", "Expected", "Actual", "Failed Checks", "Alignment"]].to_string(index=False))
print("All report alignments PASS:", all_report_alignments_pass)

=== REPORT ALIGNMENT ===
Experiment                    Expected                      Actual Failed Checks Alignment
     Exp_A GSN + sequence-wise Min-Max GSN + sequence-wise Min-Max                    PASS
     Exp_B GSN + sequence-wise Z-Score GSN + sequence-wise Z-Score                    PASS
     Exp_C TBSN + channel-wise Min-Max TBSN + channel-wise Min-Max                    PASS
     Exp_D TBSN + channel-wise Z-Score TBSN + channel-wise Z-Score                    PASS
All report alignments PASS: True


## 8. Four-branch reconciliation, provenance, gates, run manifest and final status
Mandatory gates are explicit. `number_of_excluded_trials == 0` is **not** a gate: documented exclusions yield `COMPLETED_WITH_EXCLUSIONS`, not `NOT_READY`.

In [11]:
# 8. Reconciliation table, provenance validation, gates, manifest, final status.
def _count(df, branch, status):
    return int(((df.branch_name == branch) & (df.status == status)).sum()) if len(df) else 0


def build_reconciliation() -> pd.DataFrame:
    rows = []
    vmap = {r["branch_name"]: r for r in VALIDATION_ROWS}
    for b in BRANCHES:
        n = b["branch_name"]
        v = vmap.get(n)
        comp, exc, fail = (_count(TRIAL_RECORDS_DF, n, s) for s in ("COMPLETED", "EXCLUDED", "FAILED"))
        out_dir_ = current_path(branch_out_dir(n))
        sdir = current_path(branch_scaler_dir(n))
        n_out = len(list(out_dir_.glob("*.csv"))) if out_dir_.is_dir() else 0
        scal = sorted(p.name for p in sdir.glob("*")) if sdir.is_dir() else []
        vstat = v["validation_status"] if v else "FAIL"
        if vstat == "PASS" and fail == 0:
            fstat = "READY" if exc == 0 else "COMPLETED_WITH_EXCLUSIONS"
        else:
            fstat = "NOT_READY"
        rows.append({
            "branch_name": n, "source_method": b["source_method"], "scaling_method": b["scaling_method"],
            "scaling_granularity": b["granularity"], "expected_trials": len(common_keys),
            "completed_trials": comp, "excluded_trials": exc, "failed_trials": fail,
            "expected_features": len(feature_names) if feature_names else EXPECTED_FEATURES,
            "expected_frames": EXPECTED_FRAMES,
            "output_files": n_out, "output_dir": rel(out_dir_), "mask_dir": rel(branch_mask_dir(n)),
            "scaler_files": ";".join(scal), "validation_status": vstat, "final_status": fstat,
        })
    return pd.DataFrame(rows)


def reconciliation_ok(rdf: pd.DataFrame) -> bool:
    if len(rdf) != 4 or rdf.branch_name.tolist() != EXPECTED_BRANCH_NAMES or rdf.branch_name.duplicated().any():
        return False
    ok = True
    for _, r in rdf.iterrows():
        ok &= r.expected_trials == r.completed_trials + r.excluded_trials + r.failed_trials
        ok &= r.expected_trials == len(common_keys)
        ok &= r.output_files == r.completed_trials
        ok &= r.expected_features == EXPECTED_FEATURES and r.expected_frames == EXPECTED_FRAMES
        ok &= r.scaler_files != ""
    ex_rows = EXCLUSIONS_DF.groupby("preprocessing_branch").size().to_dict() if len(EXCLUSIONS_DF) else {}
    for _, r in rdf.iterrows():
        ok &= ex_rows.get(r.branch_name, 0) == r.excluded_trials
    # exclusion is source-level: both branches of a source must exclude the same trials
    for source in SOURCES:
        sets = [set(TRIAL_RECORDS_DF[(TRIAL_RECORDS_DF.branch_name == b["branch_name"]) &
                                     (TRIAL_RECORDS_DF.status == "EXCLUDED")].trial_key)
                for b in BRANCHES if b["source_method"] == source]
        ok &= all(s == sets[0] for s in sets)
    return bool(ok)


def provenance_ok():
    problems: List[str] = []
    path = current_path(CURRENT_AUDIT_DIR, "scaled_trial_manifest.csv")
    if not path.is_file():
        return False, ["scaled_trial_manifest.csv missing in current run"]
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    required = ["run_id", "timestamp_utc", "branch_name", "source_method", "scaling_method", "scaling_granularity",
                "trial_key", "subject_id", "source_file", "feature_hash", "status"]
    for c in required:
        if c not in df.columns or (df[c] == "").any():
            problems.append(f"column '{c}' missing or has empty values")
    if problems:
        return False, problems
    if set(df.run_id) != {RUN_ID}:
        problems.append("records from a different run id present")
    if df.duplicated(["branch_name", "trial_key"]).any():
        problems.append("duplicate (branch, trial) provenance rows")
    if set(df.trial_key) != set(common_keys):
        problems.append("trial keys differ from the upstream inventory")
    if set(df.feature_hash) != {FEATURE_HASH}:
        problems.append("feature hash mismatch")
    exp_rows = len(BRANCHES) * len(common_keys)
    if len(df) != exp_rows:
        problems.append(f"expected {exp_rows} provenance rows, found {len(df)}")
    subj = trial_meta.set_index("trial_key")["subject_id"].to_dict()
    if any(subj.get(k) != s for k, s in zip(df.trial_key, df.subject_id)):
        problems.append("subject_id differs from upstream manifest")
    comp = df[df.status == "COMPLETED"]
    for col in ["output_file", "mask_file", "scaler_file"]:
        if (comp[col] == "").any():
            problems.append(f"completed rows without {col}")
        else:
            missing = [p for p in comp[col] if not (PROJECT_ROOT / p).is_file()]
            if missing:
                problems.append(f"{len(missing)} {col} paths do not exist (e.g. {missing[0]})")
    for p in comp.output_file:
        if rel(CURRENT_PROCESSED_DIR) not in p:
            problems.append("output path outside the current run directory"); break
    for p in comp.scaler_file:
        if rel(CURRENT_SCALERS_DIR) not in p:
            problems.append("scaler path outside the current run directory"); break
    sf = df[df.status.isin(["COMPLETED", "EXCLUDED"])]
    if (sf.source_file == "").any():
        problems.append("rows without source_file")
    exc_df = df[df.status == "EXCLUDED"]
    if (exc_df.exclusion_reason == "").any() or (exc_df.exclusion_stage == "").any():
        problems.append("excluded rows without documented stage/reason")
    for b in BRANCHES:
        mp = current_path(branch_scaler_dir(b["branch_name"]), "branch_metadata.json")
        if not mp.is_file():
            problems.append(f"{b['branch_name']}: branch_metadata.json missing")
        else:
            m = json.load(open(mp, encoding="utf-8"))
            if m.get("run_id") != RUN_ID or m.get("granularity") != b["granularity"]:
                problems.append(f"{b['branch_name']}: branch metadata inconsistent")
    return (not problems), problems


def to_jsonable(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return None if np.isnan(o) else float(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, (Path,)):
        return str(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)


GATE_REASONS = {
    "input_manifest_valid": "Upstream manifests/provenance/trial directories missing or inconsistent: " + "; ".join(INPUT_ERRORS[:3]),
    "four_branches_present": "The four expected branches are not defined exactly once with deterministic names.",
    "branch_definitions_valid": "Branch definitions deviate from the final design: " + "; ".join(BRANCH_DEFINITION_ERRORS[:3]),
    "feature_count_valid": "Sensor feature contract (180 = 135 lin + 45 ang, no metadata columns) violated in inputs or outputs.",
    "frame_count_valid": "A trial does not have exactly 751 frames or frame/time metadata changed.",
    "unit_tests_pass": "Scaling-core unit tests failed (see unit-test cell).",
    "current_run_outputs_exist": "Expected current-run output/mask files are missing, or unexpected files exist, for at least one branch.",
    "scalers_exist": "Scaler statistics or branch metadata missing/corrupt/inconsistent in the current run.",
    "math_validation_pass": "Formula, Min-Max range, Z-score moment or constant-channel validation failed.",
    "mask_alignment_pass": "Mask differs from source finiteness, or NaN positions are not aligned with the mask.",
    "provenance_pass": "Provenance records incomplete or inconsistent (see provenance problems printed below).",
    "reconciliation_pass": "Per-branch trial counts do not reconcile (expected != completed+excluded+failed, or files != completed).",
    "all_report_alignments_pass": "At least one of Exp_A-D deviates from the mandatory design (GSN sequence-wise, TBSN channel-wise); see REPORT_ALIGNMENT_DF.",
    "validation_uses_current_run_only": "A validation path was outside the current run directories, or current dirs coincide with legacy dirs.",
    "no_unexpected_failures": "At least one trial FAILED or a notebook stage raised an unexpected exception.",
}


def evaluate_final():
    global RECON_DF, PROVENANCE_PROBLEMS, GATES, FINAL_STATUS, FAILING_GATES, TOTALS
    RECON_DF = build_reconciliation() if (BRANCHES and VALIDATION_ROWS) else pd.DataFrame()
    prov_ok, PROVENANCE_PROBLEMS = provenance_ok() if len(TRIAL_RECORDS_DF) else (False, ["no trial records"])
    V = VALIDATION_DF if len(VALIDATION_DF) else pd.DataFrame(columns=["output_files_ok"])
    all_v = lambda col: bool(len(V) == 4 and col in V.columns and V[col].all())
    totals_failed = int((TRIAL_RECORDS_DF.status == "FAILED").sum()) if len(TRIAL_RECORDS_DF) else 0
    totals_excl = int((TRIAL_RECORDS_DF.status == "EXCLUDED").sum()) if len(TRIAL_RECORDS_DF) else 0
    totals_comp = int((TRIAL_RECORDS_DF.status == "COMPLETED").sum()) if len(TRIAL_RECORDS_DF) else 0
    TOTALS = {"expected": len(common_keys) * len(BRANCHES), "completed": totals_comp,
              "excluded": totals_excl, "failed": totals_failed}
    paths_ok = (len(ACCESSED_CURRENT_PATHS) > 0
                and all(Path(p).resolve() == CURRENT_PROCESSED_DIR.resolve() or CURRENT_PROCESSED_DIR.resolve() in Path(p).resolve().parents
                        or CURRENT_SCALERS_DIR.resolve() in Path(p).resolve().parents for p in ACCESSED_CURRENT_PATHS)
                and CURRENT_PROCESSED_DIR.resolve() != LEGACY_PROCESSED_DIR.resolve()
                and CURRENT_SCALERS_DIR.resolve() != LEGACY_SCALERS_DIR.resolve()
                and RUN_ID in CURRENT_PROCESSED_DIR.name and RUN_ID in CURRENT_SCALERS_DIR.name)
    GATES = {
        "input_manifest_valid": INPUT_OK,
        "four_branches_present": FOUR_BRANCHES_PRESENT,
        "branch_definitions_valid": BRANCH_DEFS_OK,
        "feature_count_valid": INPUT_OK and all_v("feature_count_ok"),
        "frame_count_valid": INPUT_OK and all_v("frame_count_ok"),
        "unit_tests_pass": UNIT_TESTS_PASS,
        "current_run_outputs_exist": all_v("output_files_ok"),
        "scalers_exist": all_v("scaler_ok") and all_v("scaler_files_exist"),
        "math_validation_pass": all_v("math_ok"),
        "mask_alignment_pass": all_v("mask_alignment_ok"),
        "provenance_pass": prov_ok,
        "reconciliation_pass": bool(len(RECON_DF)) and reconciliation_ok(RECON_DF),
        "all_report_alignments_pass": bool(all_report_alignments_pass),
        "validation_uses_current_run_only": paths_ok,
        "no_unexpected_failures": totals_failed == 0 and not STAGE_ERRORS,
    }
    FAILING_GATES = [k for k, v in GATES.items() if not v]
    FINAL_STATUS = determine_status(GATES, totals_failed, totals_excl)


def determine_status(gates: dict, failed: int, excluded: int) -> str:
    if all(gates.values()) and failed == 0:
        return "READY" if excluded == 0 else "COMPLETED_WITH_EXCLUSIONS"
    return "NOT_READY"


def build_run_manifest(status: str, gates: dict) -> dict:
    branches = []
    for b in BRANCHES:
        n = b["branch_name"]
        d = TRIAL_RECORDS_DF[TRIAL_RECORDS_DF.branch_name == n] if len(TRIAL_RECORDS_DF) else TRIAL_RECORDS_DF
        branches.append({
            **{k: b[k] for k in ["experiment", "branch_name", "source_method", "scaling_method", "granularity", "stats_definition"]},
            "output_dir": rel(branch_out_dir(n)), "mask_dir": rel(branch_mask_dir(n)),
            "scaler_dir": rel(branch_scaler_dir(n)),
            "scaler_files": [rel(branch_scaler_dir(n) / scaler_filename(b)), rel(branch_scaler_dir(n) / "branch_metadata.json")],
            "trial_file_pattern": rel(branch_out_dir(n)) + "/{trial_key}.csv",
            "mask_file_pattern": rel(branch_mask_dir(n)) + "/{trial_key}.npy",
            "completed_trials": int((d.status == "COMPLETED").sum()) if len(d) else 0,
            "excluded_trials": int((d.status == "EXCLUDED").sum()) if len(d) else 0,
            "failed_trials": int((d.status == "FAILED").sum()) if len(d) else 0,
            "completed_trial_keys": sorted(d[d.status == "COMPLETED"].trial_key.tolist()) if len(d) else [],
            "excluded_trial_keys": sorted(d[d.status == "EXCLUDED"].trial_key.tolist()) if len(d) else [],
            "failed_trial_keys": sorted(d[d.status == "FAILED"].trial_key.tolist()) if len(d) else [],
        })
    return {
        "run_id": RUN_ID, "timestamp_utc": RUN_TIMESTAMP,
        "notebook": NOTEBOOK_ID, "notebook_version": NOTEBOOK_VERSION,
        "granularity_profile": DESIGN_LABEL,
        "final_status": status,
        "gates": gates, "failing_gates": [k for k, v in gates.items() if not v],
        "counts": {"expected_trials_per_branch": len(common_keys), "n_branches": len(BRANCHES),
                   "expected_trial_outputs": len(common_keys) * len(BRANCHES),
                   "completed": TOTALS["completed"], "excluded": TOTALS["excluded"], "failed": TOTALS["failed"]},
        "contract": {"n_features": len(feature_names), "n_linear": N_LINEAR, "n_angular": N_ANGULAR,
                     "n_frames": EXPECTED_FRAMES, "feature_hash_sha256": FEATURE_HASH,
                     "feature_names": feature_names, "metadata_columns": META_COLUMNS,
                     "trial_csv_columns": META_COLUMNS + feature_names,
                     "mask_convention": "M=1 finite observation, M=0 missing/invalid; uint8 .npy of shape (751,180); "
                                        "NaN in X <=> M=0",
                     "missing_value_policy": "NaN preserved; no zero-fill; no imputation in Notebook 08",
                     "constant_channel_rule": "valid values -> 0; NaN preserved",
                     "zscore_std": "population std (ddof=0)"},
        "paths": {"project_root": str(PROJECT_ROOT), "processed_dir": rel(CURRENT_PROCESSED_DIR),
                  "scalers_dir": rel(CURRENT_SCALERS_DIR), "masks_dir": rel(CURRENT_MASKS_DIR),
                  "audit_dir": rel(CURRENT_AUDIT_DIR),
                  "trial_manifest": rel(CURRENT_AUDIT_DIR / "scaled_trial_manifest.csv"),
                  "exclusion_table": rel(CURRENT_AUDIT_DIR / "excluded_trials.csv"),
                  "failure_table": rel(CURRENT_AUDIT_DIR / "failed_trials.csv"),
                  "missingness_audit": rel(CURRENT_AUDIT_DIR / "input_missingness_audit.csv"),
                  "trial_metadata_copy": rel(CURRENT_AUDIT_DIR / "trial_metadata.csv"),
                  "run_manifest": rel(RUN_MANIFEST_PATH)},
        "upstream": {"gsn_dir": rel(GSN_DIR), "tbsn_dir": rel(TBSN_DIR), "gsn_manifest": rel(GSN_MANIFEST),
                     "tbsn_manifest": rel(TBSN_MANIFEST), "feature_provenance": rel(PROVENANCE),
                     "labels_metadata_reference": rel(GSN_MANIFEST)},
        "branches": branches,
        "stage_errors": STAGE_ERRORS,
    }


def write_manifest(status: str, gates: dict) -> bool:
    m = build_run_manifest(status, gates)
    with open(RUN_MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(m, f, indent=2, default=to_jsonable)
    back = json.load(open(RUN_MANIFEST_PATH, encoding="utf-8"))
    return (back["run_id"] == RUN_ID and back["final_status"] == status and len(back["branches"]) == len(BRANCHES)
            and back["contract"]["n_features"] == len(feature_names) and back["gates"] == json.loads(json.dumps(gates)))


RECON_DF = pd.DataFrame(); PROVENANCE_PROBLEMS = []; GATES = {}; FINAL_STATUS = "NOT_READY"; FAILING_GATES = []
TOTALS = {"expected": 0, "completed": 0, "excluded": 0, "failed": 0}

try:
    evaluate_final()
    gates_pre = dict(GATES)
    manifest_ok = write_manifest(FINAL_STATUS, gates_pre)
    GATES["run_manifest_written_and_readable"] = manifest_ok
    FAILING_GATES = [k for k, v in GATES.items() if not v]
    status_after = determine_status(GATES, TOTALS["failed"], TOTALS["excluded"])
    if status_after != FINAL_STATUS or not manifest_ok:
        FINAL_STATUS = status_after
        GATES["run_manifest_written_and_readable"] = write_manifest(FINAL_STATUS, GATES)
        FAILING_GATES = [k for k, v in GATES.items() if not v]
        FINAL_STATUS = determine_status(GATES, TOTALS["failed"], TOTALS["excluded"])
except Exception:
    STAGE_ERRORS["finalization"] = traceback.format_exc()
    print(STAGE_ERRORS["finalization"])
    GATES["no_unexpected_failures"] = False
    FAILING_GATES = [k for k, v in GATES.items() if not v] or ["no_unexpected_failures"]
    FINAL_STATUS = "NOT_READY"
    try:
        write_manifest(FINAL_STATUS, GATES)
    except Exception:
        STAGE_ERRORS["manifest_write"] = traceback.format_exc()

if len(RECON_DF):
    RECON_DF.to_csv(current_path(CURRENT_AUDIT_DIR, "branch_reconciliation.csv"), index=False)
with open(current_path(CURRENT_AUDIT_DIR, "final_scaling_audit.json"), "w", encoding="utf-8") as f:
    json.dump({"run_id": RUN_ID, "final_status": FINAL_STATUS, "gates": GATES, "failing_gates": FAILING_GATES,
               "totals": TOTALS, "provenance_problems": PROVENANCE_PROBLEMS, "unit_tests": UNIT_TEST_RESULTS,
               "stage_seconds": STAGE_SECONDS}, f, indent=2, default=to_jsonable)

if FINAL_STATUS in ("READY", "COMPLETED_WITH_EXCLUSIONS"):
    with open(LATEST_RUN_POINTER, "w", encoding="utf-8") as f:     # pointer only; run directories are never overwritten
        json.dump({"run_id": RUN_ID, "final_status": FINAL_STATUS, "run_manifest": rel(RUN_MANIFEST_PATH),
                   "timestamp_utc": RUN_TIMESTAMP}, f, indent=2)

print("=== FOUR-BRANCH RECONCILIATION ===")
if len(RECON_DF):
    print(RECON_DF.drop(columns=["output_dir", "mask_dir"]).to_string(index=False))
print("\n=== GATES ===")
for k, v in GATES.items():
    print(f"{'PASS' if v else 'FAIL'}  {k}")
if PROVENANCE_PROBLEMS and not GATES.get("provenance_pass", True):
    print("\nProvenance problems:", PROVENANCE_PROBLEMS)

=== FOUR-BRANCH RECONCILIATION ===
      branch_name source_method scaling_method scaling_granularity  expected_trials  completed_trials  excluded_trials  failed_trials  expected_features  expected_frames  output_files                                      scaler_files validation_status final_status
 exp_A_gsn_minmax           GSN        Min-Max       sequence-wise              881               881                0              0                180              751           881 branch_metadata.json;sequence_wise_statistics.csv              PASS        READY
 exp_B_gsn_zscore           GSN        Z-Score       sequence-wise              881               881                0              0                180              751           881 branch_metadata.json;sequence_wise_statistics.csv              PASS        READY
exp_C_tbsn_minmax          TBSN        Min-Max        channel-wise              881               881                0              0                180              751 

In [12]:
# 9. FINAL REPORT
bar = "=" * 60
yn = lambda g: "PASS" if GATES.get(g) else "FAIL"
print(bar); print("NOTEBOOK 08 FINAL SCALING AUDIT"); print(f"run_id = {RUN_ID}"); print(bar, "\n")
for _, r in REPORT_ALIGNMENT_DF.iterrows():
    print(f"{r.Experiment}: {r['Preprocessing Source']:<4s} + {r['Scaling Granularity'].capitalize():<13s} {r.Scaler:<8s} {r.Alignment}")
print()
rows = [("751-frame contract", "frame_count_valid"), ("180-feature contract", "feature_count_valid"),
        ("NaN/mask preservation", "mask_alignment_pass"), ("Math validation", "math_validation_pass"),
        ("Scaling-core unit tests", "unit_tests_pass"), ("Current-run validation", "validation_uses_current_run_only"),
        ("Outputs / scalers exist", "current_run_outputs_exist"), ("Provenance", "provenance_pass"),
        ("Reconciliation", "reconciliation_pass"), ("Report alignment", "all_report_alignments_pass")]
for text, g in rows:
    print(f"{text + ':':<28s}{yn(g)}")
# imputation guard is structural: every NaN position is aligned with M=0 and no value is filled
print(f"{'No Notebook-08 imputation:':<28s}{'PASS' if GATES.get('mask_alignment_pass') and GATES.get('unit_tests_pass') else 'FAIL'}")
print(f"\nTrials per branch: {len(common_keys)} | completed outputs: {TOTALS['completed']} | "
      f"excluded: {TOTALS['excluded']} | failed: {TOTALS['failed']} | features=180 | frames=751")
if len(INPUT_MISSINGNESS_DF):
    ok_ = INPUT_MISSINGNESS_DF[INPUT_MISSINGNESS_DF.status != "FAILED"]
    print(f"Source trials with missing values (kept as NaN, not excluded): "
          f"GSN={int(((ok_.source=='GSN') & (ok_.n_missing>0)).sum())}, TBSN={int(((ok_.source=='TBSN') & (ok_.n_missing>0)).sum())}; "
          f"Inf cells treated invalid: {int(ok_.n_inf.sum())}")
if len(EXCLUSIONS_DF):
    print("\nExclusions (stage / reason):")
    for (br, st, rs), n in EXCLUSIONS_DF.groupby(["preprocessing_branch", "exclusion_stage", "exclusion_reason"]).size().items():
        print(f"  {br}: {n} trial(s) | {st} | {rs}")
    print(f"  full log: {rel(CURRENT_AUDIT_DIR / 'excluded_trials.csv')}")
print("\n" + bar); print(f"FINAL STATUS: {FINAL_STATUS}"); print(bar)
if FINAL_STATUS == "NOT_READY":
    print("Failing gates and actionable reasons:")
    for g in FAILING_GATES:
        print(f"  - {g}: {GATE_REASONS.get(g, 'see audit files')}")
    if STAGE_ERRORS:
        print("  Stage exceptions:", ", ".join(STAGE_ERRORS))
    for _, r in (VALIDATION_DF[VALIDATION_DF.first_errors != ""].iterrows() if len(VALIDATION_DF) else []):
        print(f"  [{r.branch_name}] {r.first_errors}")
    if len(FAILURES_DF):
        print(f"  Failed trials ({len(FAILURES_DF)} rows; {rel(CURRENT_AUDIT_DIR / 'failed_trials.csv')}):")
        for _, r in FAILURES_DF.head(10).iterrows():
            print(f"    {r.trial_key} [{r.branch_name}] {r.failure_stage}: {str(r.failure_reason)[:160]}")
    print("Do NOT use this run as input to Notebook 09. Fix the cause and re-run (a new run id is created).")
else:
    print(f"Notebook 09 entry point: {rel(RUN_MANIFEST_PATH)}  (pointer: {rel(LATEST_RUN_POINTER)})")
NOTEBOOK_08_FINAL_STATUS = FINAL_STATUS

NOTEBOOK 08 FINAL SCALING AUDIT
run_id = 20261009_142420_27f01a

Exp_A: GSN  + Sequence-wise Min-Max  PASS
Exp_B: GSN  + Sequence-wise Z-Score  PASS
Exp_C: TBSN + Channel-wise  Min-Max  PASS
Exp_D: TBSN + Channel-wise  Z-Score  PASS

751-frame contract:         PASS
180-feature contract:       PASS
NaN/mask preservation:      PASS
Math validation:            PASS
Scaling-core unit tests:    PASS
Current-run validation:     PASS
Outputs / scalers exist:    PASS
Provenance:                 PASS
Reconciliation:             PASS
Report alignment:           PASS
No Notebook-08 imputation:  PASS

Trials per branch: 881 | completed outputs: 3524 | excluded: 0 | failed: 0 | features=180 | frames=751
Source trials with missing values (kept as NaN, not excluded): GSN=0, TBSN=0; Inf cells treated invalid: 0

FINAL STATUS: READY
Notebook 09 entry point: data/processed/scaled_features_run_20261009_142420_27f01a/run_manifest.json  (pointer: data/processed/LATEST_SCALING_RUN.json)


## Output contract (current run only)

```
data/processed/scaled_features_run_<RUN_ID>/
  run_manifest.json                      <- Notebook 09 entry point (machine-readable)
  <branch_name>/<trial_key>.csv          <- frame, time, <180 sensor features>; NaN where missing
  _masks/<branch_name>/<trial_key>.npy   <- uint8 mask (751 x 180), 1 = finite, 0 = missing/invalid
  _audit/
    scaled_trial_manifest.csv            <- per branch x trial provenance (status, paths, subject, hashes)
    excluded_trials.csv                  <- structured exclusion table (header-only if none)
    failed_trials.csv                    <- software/contract failures (header-only if none)
    input_missingness_audit.csv          <- per source x trial missingness (preserved, not excluded)
    branch_definitions.csv, branch_reconciliation.csv, validation_results.csv
    constant_or_zero_variance_features.csv, trial_metadata.csv, final_scaling_audit.json
data/metadata/scalers_run_<RUN_ID>/<branch_name>/
  channelwise_statistics.csv | sequence_wise_statistics.csv, branch_metadata.json
data/processed/LATEST_SCALING_RUN.json   <- pointer to the latest non-NOT_READY run (the only file ever overwritten)
```

`run_manifest.json` contains: run id, timestamp, notebook version, granularity profile, final status, all gates, counts, the 180-feature list and its SHA-256, frame/feature counts, mask convention, per-branch output/mask/scaler locations and trial-key lists, and references to the upstream manifests (labels/metadata) and the metadata copy. Labels and subject IDs are never altered.

**Runtime.** Writing ~3,500 trial CSVs dominates; expect on the order of 20–30 minutes for 880 trials.

**NOT_READY** means a genuine pipeline/validation failure (never "some trials had NaN" and never "some trials were excluded"); the final report lists the failing gates and reasons.